# Qwen3-8B 우선순위 1: 선택적 라우팅 실험 노트북

현재 최고권 제출본들을 다시 생성하지 않고, **샘플 특성에 따라 다른 강한 제출본을 선택해서 섞는 라우팅 실험**을 수행합니다.

생성 파일:
- `submit_20_route_len_v1.csv`
- `submit_21_route_complex_v1.csv`
- `submit_22_route_hybrid_v1.csv`

핵심 아이디어:
1. 짧은 대화는 `abstract_short` 사용
2. 복잡한 대화는 MBR 계열 사용
3. 중간 구간은 현재 최고점 `short_clean_v3` 중심으로 유지


In [ ]:
import sys, pandas as pd
print('python', sys.version)
print('pandas', pd.__version__)


In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path
import re
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
DATA_DIR = Path("/root/upstage-nlp-nlp/code/data")
MANUAL_DIR = Path("/root/upstage-nlp-nlp/code/prediction/qwen3_response_only_best_strategy_8b/manual_submissions")

CANDIDATES = {
    "short_clean_v3": MANUAL_DIR / "submit_12_top1_absShort_clean_v3.csv",
    "abstract_short": MANUAL_DIR / "submit_05_top1_abstract_short.csv",
    "mbr_clean_refine": MANUAL_DIR / "submit_16_top1_mbr_clean_refine.csv",
    "mbr_short_best3": MANUAL_DIR / "submit_13_top1_mbr_absShort_best3.csv",
}

OUT_FILES = {
    "route_len_v1": MANUAL_DIR / "submit_20_route_len_v1.csv",
    "route_complex_v1": MANUAL_DIR / "submit_21_route_complex_v1.csv",
    "route_hybrid_v1": MANUAL_DIR / "submit_22_route_hybrid_v1.csv",
}

# ============================================================
# 유틸
# ============================================================
def normalize_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|.*?\|>", "", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = text.replace("/no_think", " ")
    text = re.sub(r"^요약\s*:\s*", "", text).strip()
    text = re.sub(r"#\s*Person\s*(\d+)\s*#", r"#Person\1#", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_submission(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"missing candidate csv: {path}")
    df = pd.read_csv(path)
    if "fname" not in df.columns or "summary" not in df.columns:
        raise ValueError(f"invalid csv format: {path}")
    df = df[["fname", "summary"]].copy()
    df["summary"] = df["summary"].map(normalize_text)
    return df


def extract_features(dialogue: str) -> dict:
    text = str(dialogue)
    chars = len(text)
    lines = len([x for x in re.split(r"\n+", text) if x.strip()])
    person_tags = re.findall(r"#Person\d+#", text)
    turns = len(person_tags)
    questions = text.count("?") + text.count("？")
    commas = text.count(",") + text.count("，")
    periods = text.count(".") + text.count("!") + text.count(";")
    punct = questions + commas + periods
    return {
        "chars": chars,
        "lines": lines,
        "turns": turns,
        "questions": questions,
        "commas": commas,
        "punct": punct,
    }


def route_len_v1(feat: dict) -> str:
    if feat["chars"] <= 260 or feat["turns"] <= 4:
        return "abstract_short"
    if feat["chars"] >= 900 or feat["turns"] >= 12:
        return "mbr_clean_refine"
    return "short_clean_v3"


def route_complex_v1(feat: dict) -> str:
    if feat["turns"] >= 14 or feat["lines"] >= 10 or feat["punct"] >= 12:
        return "mbr_short_best3"
    if feat["chars"] <= 220 and feat["turns"] <= 4:
        return "abstract_short"
    return "short_clean_v3"


def route_hybrid_v1(feat: dict) -> str:
    if feat["chars"] <= 240 and feat["turns"] <= 4:
        return "abstract_short"
    if feat["chars"] >= 850 or feat["turns"] >= 11:
        if feat["punct"] >= 10 or feat["lines"] >= 8:
            return "mbr_clean_refine"
        return "short_clean_v3"
    if feat["questions"] >= 2 or feat["commas"] >= 3:
        return "mbr_short_best3"
    return "short_clean_v3"


def save_submission(base_df: pd.DataFrame, summaries, out_path: Path):
    out = pd.DataFrame({"fname": base_df["fname"], "summary": summaries})
    out.to_csv(out_path, index=False)
    print("saved:", out_path)


# ============================================================
# 로딩
# ============================================================
test_df = pd.read_csv(DATA_DIR / "test.csv")
base = test_df[["fname", "dialogue"]].copy()

loaded = {name: load_submission(path) for name, path in CANDIDATES.items()}
for name, df in loaded.items():
    if len(df) != len(base):
        raise ValueError(f"length mismatch: {name} -> {len(df)} vs test {len(base)}")
    if df["fname"].tolist() != base["fname"].tolist():
        raise ValueError(f"fname order mismatch: {name}")
    base[name] = df["summary"].tolist()

# ============================================================
# 라우팅 제출본 생성
# ============================================================
route_functions = {
    "route_len_v1": route_len_v1,
    "route_complex_v1": route_complex_v1,
    "route_hybrid_v1": route_hybrid_v1,
}

all_previews = []
for route_name, route_fn in route_functions.items():
    selected = []
    route_keys = []
    counts = {k: 0 for k in CANDIDATES.keys()}

    for row in base.itertuples(index=False):
        feat = extract_features(row.dialogue)
        key = route_fn(feat)
        counts[key] += 1
        route_keys.append(key)
        selected.append(getattr(row, key))

    save_submission(base, selected, OUT_FILES[route_name])
    print(route_name, "route counts:", counts)

    preview = pd.DataFrame({
        "fname": base["fname"].head(8),
        "route": route_keys[:8],
        "summary": selected[:8],
    })
    preview["submission"] = route_name
    all_previews.append(preview)

print("\n===== preview =====")
print(pd.concat(all_previews, ignore_index=True))
